# `pep_compass.analysis.reader` reference notebook

Runnable reference covering every public entry point of `analysis.reader`: `ExperimentReader`, `ExperimentSelection`, the shared `ExperimentDataset` facade, `RunReplay`, and the `MetricsStore` analysis cache. 

> It serves as a starting point when writing a new analysis notebook or test.

Runs against the real result directory `experiments/results/reference_lebo/run` checked into this repository.

> Those are results of standard lebo experiment
> 
> ```uv run --extra cu118 pep-compass run assets/experiments/configs/reference_lebo.yaml ``` 

## Setup

In [1]:
from pathlib import Path

import pandas as pd

from pep_compass.analysis.reader import ExperimentReader

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)


def _repository_root(start: Path) -> Path:
    """Walk upward from `start` to the directory containing `pyproject.toml`.

    Kernel working directories vary by IDE/session, so paths in this notebook
    are anchored to the repository root instead of `Path.cwd()`.
    """
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError(f"pyproject.toml not found above {start}")


REPO_ROOT = _repository_root(Path.cwd())
RESULTS_ROOT = REPO_ROOT / "experiments/results/reference_lebo/run"
assert RESULTS_ROOT.exists(), RESULTS_ROOT

AssertionError: /home/max/repositories/pep-compass/experiments/results/reference_lebo/run

## 1. Discovery

`ExperimentReader(path)` accepts a collection root, an experiment directory, or a single tracking-run directory, and discovers runs lazily — no tracking tables are loaded yet. It also opens a `.pep_compass_analysis.sqlite` cache next to the opened root (created on first access, see §5).

In [2]:
reader = ExperimentReader(RESULTS_ROOT)

print(f"root: {reader.root}")
print(f"experiments: {[e.name for e in reader.experiments]}")
print(f"runs: {len(reader.runs)}")
reader.runs[0]

root: /home/max/repositories/pep-compass/experiments/results/reference_lebo/run
experiments: ['run']
runs: 1


ExperimentRun(run_id='run_00000', experiment='run', grid_id='variant_00000', tracking_path=PosixPath('/home/max/repositories/pep-compass/experiments/results/reference_lebo/run/variants/variant_00000/runs/run_00000/tracking'))

In [3]:
# Run metadata is manifest/result-payload metadata (exact fields depend on the
# discovery path — versioned runtime results vs. legacy grid manifests), used by
# `select(...)` below.|
reader.runs[0].metadata

{'schema_version': '2',
 'status': 'completed',
 'run_id': 'run_00000',
 'run_index': 0,
 'task_id': 'task_00000',
 'variant_id': 'variant_00000',
 'seed': 1234,
 'sequence': 'FLYKWWIRIGRLKL',
 'candidate_count': 32,
 'best_score': 3.760430097579956,
 'objective_name': 'apex',
 'objective_direction': 'minimize',
 'experiment': 'run',
 'grid_id': 'variant_00000',
 'name': 'FLYKWWIRIGRLKL',
 'method': 'composable'}

## 2. Selecting runs

`reader.select(**selectors)` (equivalently `ExperimentSelection.select(...)`) filters *runs* by manifest/resolved-config metadata: `experiments`, `methods`, `grid_ids`, `peptides`, `seeds`, `parameters`. Selections are immutable and lazy — nothing is read from disk until you call `.collect`/`.scan`/`.count_rows`.

> Remark for more functionality we will be adding methods into `ExperimentSelection` in `src/pep_compass/analysis/reader/selection.py`. 

In [4]:
all_runs = reader.select()
print(f"all runs: {len(all_runs.runs)}")

by_seed = reader.select(seeds=[1234])
print(f"seed=1234 runs: {len(by_seed.runs)}")

by_peptide = reader.select(peptides=["FLYKWWIRIGRLKL"])
print(f"peptide=FLYKWWIRIGRLKL runs: {len(by_peptide.runs)}")

# `parameters` matches arbitrary run-metadata keys, not only resolved-config values.
by_parameter = reader.select(parameters={"objective_name": ["apex"]})
print(f"objective_name=apex runs: {len(by_parameter.runs)}")

# Selections chain: `.select(...)` on a selection narrows it further.|
narrowed = all_runs.select(seeds=[1234]).select(peptides=["FLYKWWIRIGRLKL"])
print(f"chained selection runs: {len(narrowed.runs)}")

all runs: 1
seed=1234 runs: 1
peptide=FLYKWWIRIGRLKL runs: 1
objective_name=apex runs: 1
chained selection runs: 1


`.rows(**filters)` adds deferred **row**-level filters (as opposed to run-level selectors above). They apply only to tables whose source columns contain the filter key, and only when the table is actually scanned.

In [5]:
first_candidate_only = reader.select().rows(candidate_index=0)
first_candidate_only.collect("candidates")

,candidate_index,experiment,grid_id,method,seed,name
0,0,run,variant_00000,composable,1234,FLYKWWIRIGRLKL
1,0,run,variant_00000,composable,1234,FLYKWWIRIGRLKL
2,0,run,variant_00000,composable,1234,FLYKWWIRIGRLKL
3,0,run,variant_00000,composable,1234,FLYKWWIRIGRLKL
4,0,run,variant_00000,composable,1234,FLYKWWIRIGRLKL
5,0,run,variant_00000,composable,1234,FLYKWWIRIGRLKL
6,0,run,variant_00000,composable,1234,FLYKWWIRIGRLKL
7,0,run,variant_00000,composable,1234,FLYKWWIRIGRLKL
8,0,run,variant_00000,composable,1234,FLYKWWIRIGRLKL
9,0,run,variant_00000,composable,1234,FLYKWWIRIGRLKL


## 3. Reading tables

Logical table names are registered in `data_schemas.registry.SCHEMAS` (`steps`, `candidates`, `trajectory_points`, `local_enumerations`, `stability`, plus BO-loop tables not produced by this reference run). `.collect` materializes a table across all selected runs; `.scan` yields it chunk by chunk for tables too large to hold in memory; `.count_rows` counts without materializing; `.paths` returns the underlying per-run CSV paths.

In [7]:
selection = reader.select()

print(f"candidates paths: {selection.paths('candidates')}")
print(f"candidates row count: {selection.count_rows('candidates')}")

candidates = selection.collect("candidates")
candidates.head()

candidates paths: (PosixPath('/home/max/repositories/pep-compass/experiments/results/reference_lebo/run/variants/variant_00000/runs/run_00000/tracking/candidates.csv'),)
candidates row count: 32


,execution_id,candidate_index,sequence,latent_origin,fields,experiment,grid_id,method,seed,name
0,407,0,YLYWWWPRPGRLKL,"[-1.8529995679855347,0.34533384442329407,-1.60...","{""mutation.options"":""{0: [5, 20], 3: [9, 19], ...",run,variant_00000,composable,1234,FLYKWWIRIGRLKL
1,407,1,YLYWWWIRPGRLWL,"[-1.8529995679855347,0.34533384442329407,-1.60...","{""mutation.options"":""{0: [5, 20], 3: [9, 19], ...",run,variant_00000,composable,1234,FLYKWWIRIGRLKL
2,407,2,YLYWWWPRIGRLKL,"[-1.8529995679855347,0.34533384442329407,-1.60...","{""mutation.options"":""{0: [5, 20], 3: [9, 19], ...",run,variant_00000,composable,1234,FLYKWWIRIGRLKL
3,1614,0,FLYWWWIRIGRLWL,"[-1.8529995679855347,0.34533384442329407,-1.60...","{""oracle.apex.score"":4.640906810760498,""trust_...",run,variant_00000,composable,1234,FLYKWWIRIGRLKL
4,1614,1,YLYKWWIRPGRLWL,"[-1.8529995679855347,0.34533384442329407,-1.60...","{""oracle.apex.score"":5.9991888999938965,""trust...",run,variant_00000,composable,1234,FLYKWWIRIGRLKL


In [8]:
# `.scan` is the memory-bounded counterpart of `.collect`, useful for large tables.
total_rows = 0
for chunk in selection.scan("trajectory_points", chunk_size=1000):
    total_rows += len(chunk)
print(f"trajectory_points rows via scan: {total_rows}")

local_enumerations = selection.collect("local_enumerations")
local_enumerations.head()

trajectory_points rows via scan: 3300


,execution_id,run_id,variant_id,loop_indices,input_count,input_latent_start,input_latent_count,input_sequences,output_count,output_sequences_sha256,experiment,grid_id,method,seed,name
0,3,run_00000,variant_00000,[0],1,0,1,"[""FLYKWWIRIGRLKL""]",6100,6cc73c7deb6437f12a6875e3cfe7b76496de6ea08064dd...,run,variant_00000,composable,1234,FLYKWWIRIGRLKL
1,410,run_00000,variant_00000,[1],3,1,3,"[""YLYWWWPRPGRLKL"", ""YLYWWWIRPGRLWL"", ""YLYWWWPR...",18300,c58518cb311fa702c9a04a67132363fb15b206aea664c0...,run,variant_00000,composable,1234,FLYKWWIRIGRLKL
2,1617,run_00000,variant_00000,[2],3,4,3,"[""FLYWWWIRIGRLWL"", ""YLYKWWIRPGRLWL"", ""YLYWWWHR...",18300,2468d0604f0909c73ff5378b93c204f9dcfac44424de51...,run,variant_00000,composable,1234,FLYKWWIRIGRLKL
3,2824,run_00000,variant_00000,[3],3,7,3,"[""FLYKWWIRIGRLWL"", ""YLYWWWHRIGRLWL"", ""FLYWWWFR...",18300,93e19220e2f4033c6473b6811687df318f8bfe80ff3b2b...,run,variant_00000,composable,1234,FLYKWWIRIGRLKL
4,4031,run_00000,variant_00000,[4],3,10,3,"[""FLYKWWFRIGRLWL"", ""YLYKWWIRIGRLKL"", ""FLYWWWIR...",18300,f1370906b225f7869556b8a2d7f5527b498095f80b7650...,run,variant_00000,composable,1234,FLYKWWIRIGRLKL


`.specification()` returns a stable, JSON-serializable description of the selection (run ids, tracking paths, row filters, and input-file fingerprints). It is the cache key used by `MetricsStore.put_analysis`/`get_analysis` (§5).

In [9]:
specification = selection.specification()
print(f"run_ids: {specification['run_ids']}")
print(f"row_filters: {specification['row_filters']}")
print(f"input_files tracked: {len(specification['input_files'])}")

run_ids: ['run_00000']
row_filters: {}
input_files tracked: 9


## 4. Shared `ExperimentDataset` facade

`reader.dataset` exposes the same tables through a version-tagged, backend-agnostic contract (`SelectionTableHandle`), keyed by logical table name. Only tables with at least one existing file are listed.

In [11]:
print(f"schema_version: {reader.dataset.schema_version}")
print(f"runs: {len(reader.dataset.runs)}")
print(f"tables: {list(reader.dataset.tables)}")

steps_handle = reader.dataset.tables["steps"]
print(f"steps columns: {steps_handle.columns}")
print(f"steps row count: {steps_handle.count_rows()}")

next(steps_handle.scan(["step_name", "status", "duration_seconds"], chunk_size=5))

schema_version: 2
runs: 1
tables: ['steps', 'candidates', 'trajectory_points', 'local_enumerations', 'stability']
steps columns: ('execution_id', 'run_id', 'variant_id', 'step_name', 'path', 'depth', 'loop_indices', 'branch_names', 'branch_indices', 'input_size', 'output_size', 'status', 'duration_seconds', 'oracle_calls_before', 'oracle_calls_after', 'generated_candidates_before', 'generated_candidates_after', 'error')
steps row count: 13293


,step_name,status,duration_seconds
0,SorbesWalker,completed,1.266452
1,MutangGenerator,completed,0.020992
2,LevenshteinConstraint,completed,0.005051
3,Flow,completed,0.006750
4,SorbesWalker,completed,0.051476


## 5. `RunReplay` — per-run checkpoints and final results

`reader.replay(run_id)` resolves one unambiguous run and exposes final results, the exact resolved configuration, and tracking checkpoints (latents included). Use it to inspect or reconstruct one materialized run in detail.

In [12]:
run_id = reader.runs[0].run_id
replay = reader.replay(run_id)

replay.result

{'schema_version': '2',
 'status': 'completed',
 'run_id': 'run_00000',
 'run_index': 0,
 'task_id': 'task_00000',
 'variant_id': 'variant_00000',
 'seed': 1234,
 'sequence': 'FLYKWWIRIGRLKL',
 'candidate_count': 32,
 'best_score': 3.760430097579956,
 'objective_name': 'apex',
 'objective_direction': 'minimize'}

In [13]:
replay.manifest

{'configuration_sha256': '81b85b7213b8b9584bfd9ab2c94aecf09576edaa575822869f2b0c5d20ddd372',
 'cuda_version': '11.8',
 'python_version': '3.12.13',
 'replay_contract': 'Replay local enumeration from stored inputs and RNG stream; verify ordered output sequence count and SHA-256 digest.',
 'run_id': 'run_00000',
 'schema_version': '2',
 'seed': 1234,
 'source_dirty': True,
 'source_revision': 'cf4ceaae8ef39baadce6ac8ab206c366e03f5f96',
 'torch_deterministic_algorithms': False,
 'torch_version': '2.5.1+cu118',
 'variant_id': 'variant_00000'}

In [14]:
# Full resolved configuration used to produce this run (pipeline, autoencoder, execution).
resolved_configuration = replay.resolved_configuration
print(list(resolved_configuration))

['autoencoder', 'execution', 'experiment', 'pipeline', 'source_path', 'tracking']


In [15]:
replay.final_candidates

,candidate_index,sequence,oracle.apex.score
0,0,YLYWWWPRPGRLKL,7.917146
1,1,YLYWWWIRPGRLWL,6.835644
2,2,YLYWWWPRIGRLKL,6.973114
3,3,FLYWWWIRIGRLWL,4.640907
4,4,YLYKWWIRPGRLWL,5.999189
5,5,YLYWWWHRPGRLKL,7.794804
6,6,FLYKWWIRIGRLWL,3.760430
7,7,YLYWWWHRIGRLWL,6.163826
8,8,FLYWWWFRPGRLWL,6.911884
9,9,FLYKWWFRIGRLWL,4.445007


In [16]:
final_latents = replay.final_latents
print(f"final_latents shape: {tuple(final_latents.shape)}")

final_latents shape: (32, 64)


### Trajectory checkpoints

`trajectory_points`/`trajectory_latents` cover every SORBES walk point across every trajectory in the run; `.trajectory(trajectory_id)` slices out one ordered trajectory with its matching latents.

In [17]:
print(f"trajectory_points rows: {len(replay.trajectory_points)}")
print(f"trajectory_latents shape: {tuple(replay.trajectory_latents.shape)}")

trajectory_id = replay.trajectory_points["trajectory_id"].iloc[0]
points, points_latents = replay.trajectory(trajectory_id)

print(f"trajectory_id: {trajectory_id}")
print(f"points: {len(points)}, latents shape: {tuple(points_latents.shape)}")
points.head()

trajectory_points rows: 3300
trajectory_latents shape: (3300, 64)
trajectory_id: Flow/Loop/iteration[0]/Flow/LocalEnumeration/branch[seed_00000_trajectory_00000]
points: 10, latents shape: (10, 64)


,execution_id,run_id,variant_id,loop_indices,trajectory_id,trajectory_index,rng_stream_seed,trajectory_step,point_id,sequence,latent_index,adjusted_time_step
0,4,run_00000,variant_00000,"[0, 0]",Flow/Loop/iteration[0]/Flow/LocalEnumeration/b...,0,1235,0,0,FLYKWWIRIGRLKL,0,0.01
1,8,run_00000,variant_00000,"[0, 1]",Flow/Loop/iteration[0]/Flow/LocalEnumeration/b...,0,1235,0,1,FLYKWWIRIGRLKL,1,0.01
2,12,run_00000,variant_00000,"[0, 2]",Flow/Loop/iteration[0]/Flow/LocalEnumeration/b...,0,1235,0,2,FLYKWWIRIGRLKL,2,0.01
3,16,run_00000,variant_00000,"[0, 3]",Flow/Loop/iteration[0]/Flow/LocalEnumeration/b...,0,1235,0,3,FLYKWWIRIGRLKL,3,0.01
4,20,run_00000,variant_00000,"[0, 4]",Flow/Loop/iteration[0]/Flow/LocalEnumeration/b...,0,1235,0,4,FLYKWWIRIGRLKL,4,0.01


### Local-enumeration replay verification

`local_enumerations.csv` stores only the **count** and a **SHA-256 digest** of each local-enumeration output, not the full candidate list, to keep tracking tables small. `.local_enumeration_input(execution_id)` returns the exact input sequences and latent checkpoint slice that produced one execution; `.verify_local_enumeration(execution_id, reconstructed_sequences)` compares a reconstruction's ordered sequence count and digest against the stored checkpoint.

Producing a *true* reconstruction requires re-running the exact same walker/mutation-generator/filter `Step`s built from `resolved_configuration` against this input — see `tests/analysis/reader/test_reader.py::test_reader_validates_local_enumeration_replay_checkpoint` for the full pattern (there built from mock components; for real components, build them via `pep_compass.core.builder.PipelineBuilder` from `resolved_configuration["pipeline"]`). Below only demonstrates the verification mechanics with a deliberately wrong reconstruction.

In [18]:
execution_id = int(replay.local_enumerations["execution_id"].iloc[0])
input_sequences, input_latents = replay.local_enumeration_input(execution_id)

print(f"execution_id: {execution_id}")
print(f"input sequences: {input_sequences}")
print(f"input latents shape: {tuple(input_latents.shape)}")

wrong_reconstruction = replay.verify_local_enumeration(execution_id, ["not", "the", "real", "output"])
print(wrong_reconstruction)
print(f"matches: {wrong_reconstruction.matches}")

execution_id: 3
input sequences: ('FLYKWWIRIGRLKL',)
input latents shape: (1, 64)
ReplayVerification(execution_id=3, expected_count=6100, actual_count=4, expected_sha256='6cc73c7deb6437f12a6875e3cfe7b76496de6ea08064ddc70286bbbe9a1bed1f', actual_sha256='8d19ae0afb4bf1af1aca7cac40d301ab181d955a67bec512b1f5d6b7bca268c5')
matches: False


## 6. Metrics and analysis cache (`MetricsStore`)

`reader.metrics` is a `MetricsStore` backed by `<root>/.pep_compass_analysis.sqlite`. It has two independent caches: per-sequence **metric values** (`put_metrics`/`get_metrics`/`get_or_compute_metrics`) and exact **aggregate analysis results** keyed by a stable hash of the selection + parameters (`put_analysis`/`get_analysis`/`list_analyses`/`load_analysis`). `reader.cached_analyses()` is a thin wrapper over `list_analyses()`.

In [19]:
sequences = candidates["sequence"].tolist()


def compute_length(batch: list[str]) -> dict[str, int]:
    """Stand-in for an expensive per-sequence metric (e.g. an oracle call)."""
    return {sequence: len(sequence) for sequence in batch}


# First call computes and persists; a second call would hit the cache instead.
lengths = reader.metrics.get_or_compute_metrics(
    "demo_length",
    sequences,
    compute_length,
    metric_version="1",
)
print(f"computed metrics for {len(lengths)} sequences")

cached_only = reader.metrics.get_metrics("demo_length", sequences, metric_version="1")
assert cached_only == lengths

computed metrics for 32 sequences


In [20]:
analysis_frame = pd.DataFrame(
    {"sequence": list(lengths), "length": list(lengths.values())}
)

reader.metrics.put_analysis(
    "demo_sequence_lengths",
    frame=analysis_frame,
    metadata={"description": "Reference-notebook demo analysis."},
    selection=specification,
    parameters={},
)

reader.cached_analyses()

,analysis_name,analysis_version,selection_hash,parameters_hash,selection_json,parameters_json,metadata_json
0,demo_sequence_lengths,1,2611408b3f2c1e7602bd6ca247a540ccf9bd0e0e759d93...,44136fa355b3678a1146ad16f7e8649e94fb4fc21fe77e...,"{""input_files"": [{""mtime_ns"": 1786380057748607...",{},"{""description"": ""Reference-notebook demo analy..."


In [21]:
loaded_frame, loaded_metadata = reader.metrics.get_analysis(
    "demo_sequence_lengths",
    selection=specification,
    parameters={},
)
print(loaded_metadata)
loaded_frame.head()

{'description': 'Reference-notebook demo analysis.'}


,sequence,length
0,YLYWWWPRPGRLKL,14
1,YLYWWWIRPGRLWL,14
2,YLYWWWPRIGRLKL,14
3,FLYWWWIRIGRLWL,14
4,YLYKWWIRPGRLWL,14
